# Preparando o ambiente

In [1]:
# instalações
!pip install -q ultraplot cartopy salem rasterio pyproj geopandas

# importa bibliotecas
import ultraplot as uplt
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import os
import imageio
import glob
import calendar
import pandas as pd
import glob
import numpy as np
import xarray as xr
import time
import salem
from geopy.distance import geodesic
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# monta o drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/2-PESQUISA/artigo_queimadas_vanessa_2025/0_round1_revisoes'

# diretório de entrada
dir_input = f'{dir}/output/01_focos_raw'

# diretório de saída
dir_output = f'{dir}/output'

# cria pasta de saída
os.makedirs(dir_output, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.7 MB/s eta 0:00:00
Mounted at /content/drive


# Leitura dos dados da estações de medição de qualidade do ar

In [2]:
# leitura dos dados da estações de medição de qualidade do ar
df_estacoes = pd.read_excel(f'{dir}/input/Informações-estações.xlsx')
df_estacoes.iloc[6,0] = 'Campinas'
df_estacoes.iloc[10,0] = 'Cubatão'
df_estacoes.iloc[16,0] = 'Guarulhos'
df_estacoes.iloc[17,0] = 'Santo André'
#df_estacoes = df_estacoes.drop(14) # São Caetano do Sul não tem dados
df_estacoes.iloc[14,4] = '-99.9'
df_estacoes.iloc[14,5] = '-99.9'

# lista das estações que serão utilizadas
#estacoes_selecionadas = ['São José do Rio Preto', 'Jaú', 'Ribeirão Preto', 'Araçatuba', 'Jundiaí']
estacoes_selecionadas = ['São José do Rio Preto', 'Ribeirão Preto', 'Jaú', 'Jundiaí']

# seleciona estações
df_estacoes_selecionadas = df_estacoes[ df_estacoes['Estação'].isin(estacoes_selecionadas) ]

# mostra a nova tabela
df_estacoes_selecionadas

,Estação,Código,Lat,Lon,PM10 (%),O3 (%)
2,São José do Rio Preto,116,-20.78470,-49.39830,86.85,82.51
3,Ribeirão Preto,288,-21.17708,-47.81900,76.71,69.58
4,Jaú,110,-22.29862,-48.56746,91.18,84.70
5,Jundiaí,109,-23.19200,-46.89710,92.07,89.68


# Seleciona focos das estações

Enviar os dados diários de focos de calor para os meses de agosto e setembro de 2024, extraídos para as estações de:

- São José do Rio Preto
- Ribeirão Preto
- Jaú
- Jundiaí

In [3]:
#==============================================================================================#
#                          Leitura dos focos de calor de SP em Ago-Set de 2024
#==============================================================================================#
# cria uma tabela
df_2024 = pd.DataFrame()

# arquivo
file = 'focos_br_ref_2024.zip'

# leitura da tabela
df_2024 = pd.read_csv(f'{dir_input}/{file}', compression='zip', usecols=['lat', 'lon', 'data_pas', 'estado', 'municipio'])

# transforma a coluna "datahora" para o formato "datetime"
df_2024['data_pas'] = pd.to_datetime(df_2024['data_pas'])

# seta a coluna "data_pas" como o índice da tabela
df_2024.set_index('data_pas', inplace=True)

# ordena a tabela pelo índice
df_2024.sort_index(inplace=True)

# focos de calor em SP
df_2024_sp = df_2024[ df_2024['estado'] == 'SÃO PAULO' ]

# seleciona ago e set
df_2024_sp_ago_set = df_2024_sp.loc['2024-08': '2024-09']

# remover a coluna "estado"
#df_2024_sp_ago_set.drop(columns=['estado'], inplace=True)

#==============================================================================================#
#                             Distância máxima em km
#==============================================================================================#
dismax = 20.0

#==============================================================================================#
#     Função que calcula a distância entre o focos e a estação de medição de qualidade do ar
#==============================================================================================#
def calcular_focos_proximos(estacao, df_focos, distancia_maxima):

    # Calcula distâncias para todos os focos
    #Exemplo:
    #2024-08-01 17:49:00    296.199143
    #2024-08-01 17:49:00    249.717281
    distancias = df_focos.apply(lambda x: geodesic((estacao['Lat'], estacao['Lon']), (x['lat'], x['lon'])).km, axis=1)

    # Filtra focos dentro do raio.
    #Exemplo:
    #2024-08-01 17:49:00    False
    #2024-08-01 17:49:00    False
    mask = distancias <= distancia_maxima

    focos_proximos = df_focos[mask].copy()

    focos_proximos['distancia_km'] = distancias[mask]

    focos_proximos['Estação_Referencia'] = estacao['Estação']

    #print('==================================', estacao['Estação'], '==================================' )
    #print(distancias, '\n', mask, df_focos)

    return {'Estação': estacao['Estação'],
            'Quantidade': len(focos_proximos),
            'DataFrame': focos_proximos}

#==============================================================================================#
#                              Processa todas as estações
#==============================================================================================#
resultados = [ calcular_focos_proximos(linha, df_2024_sp_ago_set, dismax) for _, linha in df_estacoes_selecionadas.iterrows() ]

#==============================================================================================#
#                             Salva os CSVs e exibe resumo
#==============================================================================================#
for resultado in resultados:

    nome_sanitizado = 'focos_por_estacao_' + resultado['Estação'].replace("/", "-").replace("\\", "-")

    nome_arquivo = f'{dir_output}/{nome_sanitizado}_{int(dismax)}km.csv'

    resultado['DataFrame'].to_csv(nome_arquivo, index=True)

    print(f"Salvo {nome_arquivo} - {resultado['Quantidade']} focos encontrados")

print("Processo concluído!")

Salvo /content/drive/MyDrive/2-PESQUISA/artigo_queimadas_vanessa_2025/0_round1_revisoes/output/focos_por_estacao_São José do Rio Preto_20km.csv - 79 focos encontrados
Salvo /content/drive/MyDrive/2-PESQUISA/artigo_queimadas_vanessa_2025/0_round1_revisoes/output/focos_por_estacao_Ribeirão Preto_20km.csv - 83 focos encontrados
Salvo /content/drive/MyDrive/2-PESQUISA/artigo_queimadas_vanessa_2025/0_round1_revisoes/output/focos_por_estacao_Jaú_20km.csv - 71 focos encontrados
Salvo /content/drive/MyDrive/2-PESQUISA/artigo_queimadas_vanessa_2025/0_round1_revisoes/output/focos_por_estacao_Jundiaí_20km.csv - 41 focos encontrados
Processo concluído!


In [4]:
# mostra os focos de sp em ago-set de 2024
df_2024_sp_ago_set

,lat,lon,estado,municipio
data_pas,,,,
2024-08-01 17:49:00,-21.68418,-48.85470,SÃO PAULO,IBITINGA
2024-08-01 17:49:00,-21.68582,-48.86524,SÃO PAULO,IBITINGA
2024-08-01 17:49:00,-23.36720,-49.23908,SÃO PAULO,TEJUPÁ
2024-08-01 17:49:00,-22.89354,-49.61684,SÃO PAULO,SANTA CRUZ DO RIO PARDO
2024-08-01 17:49:00,-21.64608,-49.75835,SÃO PAULO,LINS
...,...,...,...,...
2024-09-30 17:20:00,-20.18561,-50.00773,SÃO PAULO,CARDOSO
2024-09-30 17:20:00,-20.15956,-50.03808,SÃO PAULO,PEDRANÓPOLIS
2024-09-30 17:20:00,-20.18839,-50.03463,SÃO PAULO,PEDRANÓPOLIS


In [5]:
# mostra os dados
resultados

[{'Estação': 'São José do Rio Preto',
  'Quantidade': 79,
  'DataFrame':                           lat       lon     estado              municipio  \
  data_pas                                                                    
  2024-08-03 17:34:00 -20.75981 -49.25706  SÃO PAULO               GUAPIAÇU   
  2024-08-03 17:34:00 -20.76125 -49.26792  SÃO PAULO               GUAPIAÇU   
  2024-08-03 17:34:00 -20.76930 -49.25560  SÃO PAULO               GUAPIAÇU   
  2024-08-03 17:34:00 -20.77074 -49.26646  SÃO PAULO               GUAPIAÇU   
  2024-08-04 18:13:00 -20.70638 -49.36021  SÃO PAULO                 IPIGUÁ   
  ...                       ...       ...        ...                    ...   
  2024-09-26 17:54:00 -20.83603 -49.51353  SÃO PAULO               MIRASSOL   
  2024-09-26 17:54:00 -20.80007 -49.27740  SÃO PAULO  SÃO JOSÉ DO RIO PRETO   
  2024-09-26 17:54:00 -20.80158 -49.28726  SÃO PAULO  SÃO JOSÉ DO RIO PRETO   
  2024-09-26 17:54:00 -20.80914 -49.27578  SÃO PAULO  SÃO JO

In [6]:
df = pd.read_csv(nome_arquivo)
df

,data_pas,lat,lon,estado,municipio,distancia_km,Estação_Referencia
0,2024-08-06 17:56:00,-23.24764,-46.74671,SÃO PAULO,CAMPO LIMPO PAULISTA,16.580802,Jundiaí
1,2024-08-07 16:58:00,-23.30738,-46.92079,SÃO PAULO,JUNDIAÍ,13.005983,Jundiaí
2,2024-08-08 17:39:00,-23.05400,-46.96201,SÃO PAULO,VINHEDO,16.666507,Jundiaí
3,2024-08-08 17:39:00,-23.24072,-46.79568,SÃO PAULO,CAMPO LIMPO PAULISTA,11.699638,Jundiaí
4,2024-08-08 17:39:00,-23.23924,-46.78589,SÃO PAULO,CAMPO LIMPO PAULISTA,12.528009,Jundiaí
5,2024-08-08 17:39:00,-23.25253,-46.87397,SÃO PAULO,JUNDIAÍ,7.109292,Jundiaí
6,2024-08-20 17:34:00,-23.13377,-46.89245,SÃO PAULO,JUNDIAÍ,6.466312,Jundiaí
7,2024-08-20 17:34:00,-23.14133,-46.88126,SÃO PAULO,JUNDIAÍ,5.841221,Jundiaí
8,2024-08-21 18:13:00,-23.17884,-46.77333,SÃO PAULO,CAMPO LIMPO PAULISTA,12.755352,Jundiaí
9,2024-08-21 18:13:00,-23.18567,-46.80542,SÃO PAULO,JUNDIAÍ,9.412283,Jundiaí
